In [ ]:
# Import libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle


In [ ]:
print(np.__version__)

2.0.2


In [ ]:
# Load dataset
crop = pd.read_csv("/content/Crop_recommendation.csv")

In [ ]:
# Encode label
crop_dict = {
    'rice': 1, 'maize': 2, 'jute': 3, 'cotton': 4, 'coconut': 5, 'papaya': 6,
    'orange': 7, 'apple': 8, 'muskmelon': 9, 'watermelon': 10, 'grapes': 11,
    'mango': 12, 'banana': 13, 'pomegranate': 14, 'lentil': 15, 'blackgram': 16,
    'mungbean': 17, 'mothbeans': 18, 'pigeonpeas': 19, 'kidneybeans': 20,
    'chickpea': 21, 'coffee': 22
}
crop['crop_num'] = crop['label'].map(crop_dict)
crop.drop(['label'], axis=1, inplace=True)

In [ ]:
# Features and target
X = crop.drop('crop_num', axis=1)
y = crop['crop_num']

In [ ]:
# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

In [ ]:
# Scaling (optional but recommended for models other than trees)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Train model
rfc = RandomForestClassifier()
rfc.fit(X_train, y_train)

RandomForestClassifier()

In [ ]:
# Evaluate
ypred = rfc.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, ypred))
print("Classification Report:\n", classification_report(y_test, ypred))
print("Confusion Matrix:\n", confusion_matrix(y_test, ypred))

Random Forest Accuracy: 0.9977272727272727
Classification Report:
               precision    recall  f1-score   support

           1       1.00      1.00      1.00        20
           2       1.00      1.00      1.00        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00        20
           6       1.00      1.00      1.00        20
           7       1.00      1.00      1.00        20
           8       1.00      1.00      1.00        20
           9       1.00      1.00      1.00        20
          10       1.00      1.00      1.00        20
          11       1.00      1.00      1.00        20
          12       1.00      1.00      1.00        20
          13       1.00      1.00      1.00        20
          14       1.00      1.00      1.00        20
          15       1.00      1.00      1.00        20
          16       0.95      1.00      0.98        20
          17  

In [ ]:
# Prediction function
def recommendation(N, P, K, temperature, humidity, ph, rainfall):
    features = np.array([[N, P, K, temperature, humidity, ph, rainfall]])
    transformed_features = scaler.transform(features)
    prediction = rfc.predict(transformed_features)
    return prediction[0]

In [ ]:
# Crop decoder
crop_decoder = {v: k.capitalize() for k, v in crop_dict.items()}

In [ ]:
# Test prediction
test_inputs = [
    [40, 72, 77, 17.02498456, 16.98861173, 7.485996067, 88.55123143],
    [90, 42, 43, 20.87974371, 82.00274423,
6.502985292000001,202.9355362 ],
    [10, 10, 10, 15.0, 80.0, 4.5, 10.0]
]

In [ ]:
for i, inputs in enumerate(test_inputs, 1):
    predicted = recommendation(*inputs)
    crop_name = crop_decoder.get(predicted, "Unknown")
    print(f"\nTest Input {i}: Best crop to cultivate is --> {crop_name}")


Test Input 1: Best crop to cultivate is --> Chickpea

Test Input 2: Best crop to cultivate is --> Rice

Test Input 3: Best crop to cultivate is --> Muskmelon


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [ ]:
# Save model and scaler
pickle.dump(rfc, open('model.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))

Model Deployment

In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 5.1 MB/s eta 0:00:00


In [ ]:
# import gradio as gr
# import numpy as np
# import pickle

# # Load model and scaler
# model = pickle.load(open("model.pkl", "rb"))
# scaler = pickle.load(open("scaler.pkl", "rb"))

# # Crop dictionary
# crop_dict = {
#     1: "Rice", 2: "Maize", 3: "Jute", 4: "Cotton", 5: "Coconut", 6: "Papaya",
#     7: "Orange", 8: "Apple", 9: "Muskmelon", 10: "Watermelon", 11: "Grapes",
#     12: "Mango", 13: "Banana", 14: "Pomegranate", 15: "Lentil", 16: "Blackgram",
#     17: "Mungbean", 18: "Mothbeans", 19: "Pigeonpeas", 20: "Kidneybeans",
#     21: "Chickpea", 22: "Coffee"
# }

# # Prediction function
# def recommend_crop(N, P, K, temperature, humidity, ph, rainfall):
#     input_features = np.array([[N, P, K, temperature, humidity, ph, rainfall]])
#     scaled = scaler.transform(input_features)
#     prediction = model.predict(scaled)[0]
#     crop_name = crop_dict.get(prediction, "Unknown")
#     return f"🌱 Recommended Crop: **{crop_name}**"

# # Gradio interface
# inputs = [
#     gr.Number(label="Nitrogen (N)"),
#     gr.Number(label="Phosphorus (P)"),
#     gr.Number(label="Potassium (K)"),
#     gr.Number(label="Temperature (°C)"),
#     gr.Number(label="Humidity (%)"),
#     gr.Number(label="pH"),
#     gr.Number(label="Rainfall (mm)")
# ]

# demo = gr.Interface(
#     fn=recommend_crop,
#     inputs=inputs,
#     outputs="markdown",
#     title="AgroShield 🌾 Crop Recommendation System",
#     description="Enter soil and weather conditions to get the best crop recommendation using AI!"
# )

# if __name__ == "__main__":
#     demo.launch()

import gradio as gr
import pickle
import numpy as np

# Load model and scaler
model = pickle.load(open("model.pkl", "rb"))
scaler = pickle.load(open("scaler.pkl", "rb"))

# Crop dictionary
crop_dict = {
    1: "Rice", 2: "Maize", 3: "Jute", 4: "Cotton", 5: "Coconut", 6: "Papaya", 7: "Orange",
    8: "Apple", 9: "Muskmelon", 10: "Watermelon", 11: "Grapes", 12: "Mango", 13: "Banana",
    14: "Pomegranate", 15: "Lentil", 16: "Blackgram", 17: "Mungbean", 18: "Mothbeans",
    19: "Pigeonpeas", 20: "Kidneybeans", 21: "Chickpea", 22: "Coffee"
}

# Prediction function
def recommend_crop(N, P, K, temperature, humidity, ph, rainfall):
    features = np.array([[N, P, K, temperature, humidity, ph, rainfall]])
    features_scaled = scaler.transform(features)
    prediction = model.predict(features_scaled)[0]
    return f"🌾 Recommended Crop: {crop_dict.get(prediction, 'Unknown')}"

# Custom CSS for black background and white text
custom_css = """
body {
    background: linear-gradient(rgba(0,0,0,0.85), rgba(0,0,0,0.85)),
                url('https://images.unsplash.com/photo-1601004890684-d8cbf643f5f2') no-repeat center center fixed;
    background-size: cover;
    font-family: 'Segoe UI', sans-serif;
    color: #fff;
}

#logo img {
    display: block;
    margin: auto;
    width: 90px;
    margin-bottom: 10px;
}

.gradio-container {
    max-width: 700px;
    margin: auto;
    padding: 25px;
    border-radius: 20px;
    background-color: #222222dd;
    box-shadow: 0px 8px 20px rgba(0, 0, 0, 0.4);
}

.range-box {
    background-color: #333333;
    padding: 25px;
    border-radius: 15px;
    margin-bottom: 25px;
    font-size: 17px;
    color: #f1f1f1;
    line-height: 1.8;
    box-shadow: 0 0 10px rgba(0,0,0,0.2);
    font-weight: bold;
}
"""

# Input range guide (styled and separated)
range_guide = """
🔍 **Input Parameter Ranges**:

🌿 **Nitrogen (N):** 0 - 140
🌿 **Phosphorus (P):** 5 - 145
🌿 **Potassium (K):** 5 - 205
🌡️ **Temperature:** 8.0°C - 43.0°C
💧 **Humidity:** 14.0% - 100.0%
⚗️ **pH Value:** 3.5 - 9.9
🌧️ **Rainfall:** 20.0mm - 300.0mm
"""

# Gradio Interface
with gr.Blocks(css=custom_css) as demo:
    with gr.Column():
        # gr.Markdown("    ##     🌱 **AgroShield - Crop Recommendation System**     ")
        gr.HTML('<h2 style="text-align: center;">🌱 <b>AgroShield - Crop Recommendation System</b></h2>')
        gr.HTML('<div id="logo"><img src="https://img.icons8.com/office/80/plant-under-rain.png" alt="Logo"></div>')
        gr.Markdown("Predict the most suitable crop based on soil and weather inputs.")
        gr.Markdown(f'<div class="range-box">{range_guide}</div>')

        with gr.Row():
            N = gr.Number(label="Nitrogen (N)", value=50)
            P = gr.Number(label="Phosphorus (P)", value=50)
            K = gr.Number(label="Potassium (K)", value=50)

        with gr.Row():
            temp = gr.Number(label="Temperature (°C)", value=25.0)
            humidity = gr.Number(label="Humidity (%)", value=60.0)

        with gr.Row():
            ph = gr.Number(label="pH", value=6.5)
            rainfall = gr.Number(label="Rainfall (mm)", value=100.0)

        output = gr.Textbox(label="Prediction")

        submit = gr.Button("🚀 Recommend Crop")
        submit.click(fn=recommend_crop, inputs=[N, P, K, temp, humidity, ph, rainfall], outputs=output)

# Launch app
demo.launch()







It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d5a16f7dcfc0f2c375.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')